# SIP Week 2 — *with line* vs *without line* · 🔑 SOLUTION KEY (instructor copy)

Filled-in version of the student notebook. Every **✏️ Exercise** has worked code plus
grading notes. Runs on the included samples:
- `sample_calibration_moving.csv` — "with line" (moving dot), ~3500 frames
- `sample_calibration_dots.csv` — "without line" (fixed ~20 dots)
- `sample_calibration_bad.csv` — a failed calibration (for the bonus)

**Quick reference results on the samples:**
| | moving (with line) | dots (without line) | bad |
|---|---|---|---|
| rows | ~3503 | 20 | 20 |
| distinct targets | ~3503 | 20 | 20 |
| screen coverage | ~95% × 90% | ~95% × 90% | ~95% × 90% |
| corr(yaw, targetX) | +0.91 | +0.94 | -0.11 ❌ |
| corr(pitch, targetY) | -0.91 | -0.91 | +0.02 ❌ |

> Don't share this file with students — give them `SIP_Week2_Calibration_Analysis.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

## Part 1 — pandas refresher + Exercise 1 solution

In [ ]:
demo = pd.DataFrame({
    "name":  ["Ava", "Ben", "Cy", "Dia", "Eli"],
    "age":   [21, 23, 22, 24, 20],
    "score": [88, 72, 95, 60, 79],
})

# --- Exercise 1 solution ---
print(demo[["name", "score"]])                 # 1) two columns
print("\naverage age:", demo["age"].mean())    # 2) -> 22.0
print("\nunder-22:\n", demo[demo["age"] < 22]) # 3) filter
print("\nhighest score:", demo["score"].max()) # 4) -> 95

## Part 2 — Load both files + Exercise 2 solution

In [ ]:
def load_calib(path):
    """Load a calibration CSV. Returns (dataframe, metadata_dict)."""
    with open(path) as f:
        first_line = f.readline().strip()
    meta = json.loads(first_line.lstrip("#")) if first_line.startswith("#") else {}
    df = pd.read_csv(path, comment="#")
    return df, meta

WITH_LINE_PATH    = "sample_calibration_moving.csv"
WITHOUT_LINE_PATH = "sample_calibration_dots.csv"

df_line, meta_line = load_calib(WITH_LINE_PATH)
df_dots, meta_dots = load_calib(WITHOUT_LINE_PATH)

print("WITH line   :", df_line.shape, "| screen", meta_line.get("calibrationWidth"), "x", meta_line.get("calibrationHeight"))
print("WITHOUT line:", df_dots.shape, "| screen", meta_dots.get("calibrationWidth"), "x", meta_dots.get("calibrationHeight"))

In [ ]:
# --- Exercise 2 solution: explore both files ---
for name, df in [("WITH line", df_line), ("WITHOUT line", df_dots)]:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print(df[["yaw", "pitch", "roll"]].describe().round(1))
    print("missing:", df[["yaw", "pitch", "roll"]].isna().sum().to_dict())
    print()

# Grading note: the obvious difference is ROW COUNT — the moving file has thousands of
# frames (one per video frame as the dot slides), the dots file has ~20 (one per dot).
# Both should have ~0 missing values.

## Part 3 — Target pattern + Exercise 3 solution

In [ ]:
# --- Exercise 3 solution: target pattern, side by side ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(df_line["targetX"], df_line["targetY"], s=5, alpha=0.4)
ax[0].invert_yaxis(); ax[0].set_title("WITH line (moving dot)")
ax[0].set_xlabel("targetX (px)"); ax[0].set_ylabel("targetY (px)")

ax[1].scatter(df_dots["targetX"], df_dots["targetY"], s=60, alpha=0.8, color="C1")
ax[1].invert_yaxis(); ax[1].set_title("WITHOUT line (fixed dots)")
ax[1].set_xlabel("targetX (px)"); ax[1].set_ylabel("targetY (px)")
plt.tight_layout(); plt.show()

print("distinct positions  WITH line:", df_line[["targetX","targetY"]].drop_duplicates().shape[0])
print("distinct positions  WITHOUT line:", df_dots[["targetX","targetY"]].drop_duplicates().shape[0])

# Answer: the moving file fills in a continuous path (≈ as many distinct positions as
# rows); the dots file lands on ~20 fixed positions. Both cover the whole screen.

## Part 4 — Head pose vs target + Exercise 4 solution

In [ ]:
# --- Exercise 4 solution: head pose vs target for both files ---
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
for col, (name, df) in enumerate([("WITH line", df_line), ("WITHOUT line", df_dots)]):
    ax[0, col].scatter(df["yaw"], df["targetX"], s=8, alpha=0.3)
    ax[0, col].set_title(f"{name}: yaw vs targetX")
    ax[0, col].set_xlabel("yaw (deg)"); ax[0, col].set_ylabel("targetX (px)")
    ax[1, col].scatter(df["pitch"], df["targetY"], s=8, alpha=0.3, color="C1")
    ax[1, col].set_title(f"{name}: pitch vs targetY")
    ax[1, col].set_xlabel("pitch (deg)"); ax[1, col].set_ylabel("targetY (px)")
plt.tight_layout(); plt.show()

for name, df in [("WITH line", df_line), ("WITHOUT line", df_dots)]:
    print(f"{name:13s} corr(yaw,targetX)={df['yaw'].corr(df['targetX']):+.3f}  "
          f"corr(pitch,targetY)={df['pitch'].corr(df['targetY']):+.3f}")

# Answers:
# - Both files show strong tracking (|corr| ~0.9). On the samples the dots file is even
#   slightly tighter (+0.94 vs +0.91) — fewer but cleaner points.
# - The negative pitch correlation is just a sign convention (screen Y grows downward);
#   the STRENGTH is what matters, not the sign.

## Part 5 — Spot the difference + Exercise 5 solution

In [ ]:
# --- Exercise 5 solution: comparison table ---
def summarize(df, meta):
    W, H = meta["calibrationWidth"], meta["calibrationHeight"]
    return {
        "rows": len(df),
        "distinct_targets": df[["targetX", "targetY"]].drop_duplicates().shape[0],
        "x_coverage_%": round((df["targetX"].max() - df["targetX"].min()) / W * 100, 1),
        "y_coverage_%": round((df["targetY"].max() - df["targetY"].min()) / H * 100, 1),
        "corr_yaw_targetX": round(df["yaw"].corr(df["targetX"]), 3),
        "corr_pitch_targetY": round(df["pitch"].corr(df["targetY"]), 3),
    }

compare = pd.DataFrame({
    "WITH line":    summarize(df_line, meta_line),
    "WITHOUT line": summarize(df_dots, meta_dots),
})
compare

# Grading note: rows differ hugely (thousands vs ~20); coverage is similar; both have
# strong correlations. There's no single "winner" — that's the point of Part 6.

## Part 6 — Think it through (model answers)

1. **More data:** the *with-line* file has far more rows (one per video frame). More data ≠ automatically better — extra frames can be redundant or noisier; clean coverage matters more than raw count.
2. **Coverage:** both reach the screen edges (~95% × 90%). Covering edges matters because calibration must learn the head pose for *extreme* gaze directions, not just the center.
3. **Cleaner relationship:** on the samples the fixed-dots file is slightly tighter (+0.94 vs +0.91). Fixed dots let the person settle and hold still (clean sample); the moving dot is captured mid-motion, adding a little noise/lag.
4. **Following:** holding on a fixed dot is easier and steadier; chasing a moving dot adds tracking lag and overshoot — that shows up as extra scatter.
5. **Time:** the *with-line* file has time-ordered frames, so you can study motion over time (yaw vs timestamp), smoothness, reaction lag — impossible with ~20 unordered dots.
6. **Which to choose:** defensible either way. Fixed dots = clean, simple, fast. Moving = far more data + temporal info, good for studying dynamics. Reward the *reasoning*, not the choice.

**What to reward when grading:** every plot has labels + title; correct use of `comment="#"`; interpreting correlation (not just reporting it); noticing the row-count difference; thoughtful, evidence-based answers here.

## 🏁 Bonus — failed calibration (solution)

In [ ]:
# --- Bonus solution: compare a good file to the failed one ---
df_bad, meta_bad = load_calib("sample_calibration_bad.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(df_dots["yaw"], df_dots["targetX"], s=60, alpha=0.8)
ax[0].set_title(f"GOOD dots  (corr={df_dots['yaw'].corr(df_dots['targetX']):+.2f})")
ax[0].set_xlabel("yaw (deg)"); ax[0].set_ylabel("targetX (px)")
ax[1].scatter(df_bad["yaw"], df_bad["targetX"], s=60, alpha=0.8, color="crimson")
ax[1].set_title(f"BAD  (corr={df_bad['yaw'].corr(df_bad['targetX']):+.2f})")
ax[1].set_xlabel("yaw (deg)"); ax[1].set_ylabel("targetX (px)")
plt.tight_layout(); plt.show()

# Answer: the good file shows a clear rising trend (corr ~ +0.9). The bad file is a
# shapeless cloud (corr ~ 0): head pose does NOT track the target at all. Likely causes:
# tracking never locked onto the face, the person didn't actually follow the dot, or
# poor lighting/camera position. This is exactly the kind of file you'd want to discard.

## ✅ Wrap-up

Students should turn in: both target-pattern plots (Part 3), the comparison table (Part 5), written answers to Part 6, and the bonus finding. Reward labeled plots, correct loading, and **interpretation over raw numbers**.